# Chapter 10.3: Memory Management Comparison - PagedAttention vs RadixAttention

This notebook provides a deep dive into the memory management strategies of **vLLM** (PagedAttention) and **SGLang** (RadixAttention). We build visual simulators, compare cache hit rates, and analyze memory fragmentation.

## Learning Objectives
- Understand PagedAttention's block table mechanism
- Understand RadixAttention's radix tree approach
- Visualize memory layouts for both systems
- Compare prefix sharing efficiency
- Analyze memory fragmentation characteristics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part10/chapter_10.3_memory.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part10/chapter_10.3_memory.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

In [ ]:
# Setup
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, Rectangle
from matplotlib.collections import PatchCollection
import numpy as np
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Set
import random
from collections import defaultdict, OrderedDict

random.seed(42)
np.random.seed(42)
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11
print("Environment ready for memory management comparison.")

## 1. Conceptual Overview

| Aspect | PagedAttention (vLLM) | RadixAttention (SGLang) |
|--------|----------------------|------------------------|
| **Core Idea** | Virtual memory paging for KV cache | Radix tree for prefix-aware caching |
| **Unit** | Fixed-size blocks (e.g., 16 tokens) | Variable-length prefix nodes |
| **Sharing** | Copy-on-write for beam search | Automatic prefix deduplication |
| **Fragmentation** | Internal (last block waste) | Minimal (tree-structured) |
| **Lookup** | O(1) via block table | O(n) tree traversal |
| **Eviction** | Free entire sequence | LRU on tree nodes |

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle
import numpy as np

def draw_paged_vs_radix_attention():
    """PagedAttention vs RadixAttention visual comparison with same request set."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

    # --- PagedAttention (Left): Block table with hash-based lookup ---
    ax1.set_xlim(0, 10); ax1.set_ylim(0, 12); ax1.axis('off')
    ax1.set_title('PagedAttention (vLLM)\nBlock Table with Hash-Based Lookup',
                  fontsize=13, fontweight='bold', color='#4A90D9', pad=15)

    # Block table
    ax1.text(1.5, 11.2, 'Block Tables', fontsize=11, fontweight='bold', color='#333')
    requests = [
        ('Req 0: "System prompt + Hello"', ['B0','B1','B2','B3'], '#4A90D9'),
        ('Req 1: "System prompt + World"', ['B4','B5','B6','B7'], '#27AE60'),
        ('Req 2: "System prompt + Foo"',   ['B8','B9','B10','B11'], '#F39C12'),
    ]
    for i, (label, blocks, color) in enumerate(requests):
        y = 9.5 - i * 2.0
        ax1.text(0.3, y + 0.5, label, fontsize=8, color=color, fontweight='bold')
        for j, bname in enumerate(blocks):
            x = 0.3 + j * 2.2
            rect = Rectangle((x, y - 0.6), 1.8, 0.9, facecolor=color, edgecolor='white',
                              linewidth=1.5, alpha=0.8)
            ax1.add_patch(rect)
            ax1.text(x + 0.9, y - 0.15, bname, ha='center', va='center',
                     fontsize=8, fontweight='bold', color='white')

    # Physical blocks grid
    ax1.text(0.5, 3.5, 'Physical GPU Blocks (16 tokens each)', fontsize=9, fontweight='bold', color='#333')
    block_owners = ['R0','R0','R0','R0','R1','R1','R1','R1','R2','R2','R2','R2','--','--','--','--']
    block_colors_map = {'R0': '#4A90D9', 'R1': '#27AE60', 'R2': '#F39C12', '--': '#DDD'}
    for i in range(16):
        row, col = i // 4, i % 4
        x = 0.3 + col * 2.2
        y = 2.5 - row * 0.9
        owner = block_owners[i]
        rect = Rectangle((x, y), 1.8, 0.7, facecolor=block_colors_map[owner],
                          edgecolor='#333', linewidth=0.5, alpha=0.7 if owner != '--' else 0.3)
        ax1.add_patch(rect)
        ax1.text(x + 0.9, y + 0.35, f'B{i}', ha='center', va='center', fontsize=7, color='white')

    ax1.text(5, 0.2, 'No prefix sharing!\n3 copies of system prompt', ha='center',
             fontsize=10, color='#E74C3C', fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFEBEE', edgecolor='#E74C3C'))

    # --- RadixAttention (Right): Radix tree with path-based lookup ---
    ax2.set_xlim(0, 10); ax2.set_ylim(0, 12); ax2.axis('off')
    ax2.set_title('RadixAttention (SGLang)\nRadix Tree with Path-Based Lookup',
                  fontsize=13, fontweight='bold', color='#27AE60', pad=15)

    # Draw radix tree
    # Root
    root = FancyBboxPatch((3.5, 10.0), 3, 1.2, boxstyle='round,pad=0.1',
                          facecolor='#333', edgecolor='white', linewidth=2)
    ax2.add_patch(root)
    ax2.text(5, 10.6, 'Root', ha='center', va='center', fontsize=10, fontweight='bold', color='white')

    # Shared system prompt
    shared = FancyBboxPatch((2.5, 7.5), 5, 1.5, boxstyle='round,pad=0.1',
                            facecolor='#8E44AD', edgecolor='white', linewidth=2, alpha=0.9)
    ax2.add_patch(shared)
    ax2.text(5, 8.25, '"System prompt"\n(SHARED, 1 copy)', ha='center', va='center',
             fontsize=9, fontweight='bold', color='white')
    ax2.annotate('', xy=(5, 9.0), xytext=(5, 10.0),
                arrowprops=dict(arrowstyle='<-', color='#333', lw=2))

    # Three leaf branches
    leaves = [
        (0.5, 5.0, 2.5, 1.3, '"Hello"', '#4A90D9'),
        (3.75, 5.0, 2.5, 1.3, '"World"', '#27AE60'),
        (7.0, 5.0, 2.5, 1.3, '"Foo"', '#F39C12'),
    ]
    for x, y, w, h, label, color in leaves:
        leaf = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                              facecolor=color, edgecolor='white', linewidth=2, alpha=0.85)
        ax2.add_patch(leaf)
        ax2.text(x + w/2, y + h/2, f'Req suffix:\n{label}', ha='center', va='center',
                 fontsize=8, fontweight='bold', color='white')
        ax2.annotate('', xy=(x + w/2, y + h), xytext=(5, 7.5),
                    arrowprops=dict(arrowstyle='<-', color=color, lw=2,
                                    connectionstyle='arc3,rad=0'))

    # Memory savings
    ax2.text(5, 3.0, 'System prompt stored ONCE\nAll 3 requests share the prefix!',
             ha='center', fontsize=10, color='#27AE60', fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='#E8F5E9', edgecolor='#27AE60'))

    # Memory comparison bar
    ax2.text(5, 1.5, 'Memory: 1 shared + 3 suffixes\nvs 3 full copies (PagedAttention)',
             ha='center', fontsize=9, color='#555')

    plt.tight_layout()
    plt.savefig('paged_vs_radix_attention.png', dpi=150, bbox_inches='tight')
    plt.show()

draw_paged_vs_radix_attention()

In [ ]:
# Demo 1: PagedAttention Simulator

class PagedAttentionSimulator:
    """Simulates vLLM's PagedAttention memory management."""
    
    def __init__(self, num_blocks: int = 32, block_size: int = 16):
        self.num_blocks = num_blocks
        self.block_size = block_size
        self.free_blocks: List[int] = list(range(num_blocks))
        self.block_tables: Dict[int, List[int]] = {}  # seq_id -> [block_ids]
        self.block_usage: Dict[int, int] = {}  # block_id -> tokens_used
        self.ref_counts: Dict[int, int] = defaultdict(int)  # block_id -> ref_count
        self.allocation_log = []
    
    @property
    def used_blocks(self):
        return self.num_blocks - len(self.free_blocks)
    
    @property
    def utilization(self):
        if not self.block_usage:
            return 0.0
        total_slots = len(self.block_usage) * self.block_size
        used_slots = sum(self.block_usage.values())
        return used_slots / total_slots if total_slots > 0 else 0.0
    
    def allocate(self, seq_id: int, num_tokens: int) -> bool:
        """Allocate blocks for a new sequence."""
        num_blocks_needed = (num_tokens + self.block_size - 1) // self.block_size
        if len(self.free_blocks) < num_blocks_needed:
            return False
        
        blocks = []
        for _ in range(num_blocks_needed):
            block_id = self.free_blocks.pop(0)
            blocks.append(block_id)
            self.ref_counts[block_id] = 1
        
        self.block_tables[seq_id] = blocks
        
        # Track usage per block
        remaining = num_tokens
        for block_id in blocks:
            used = min(remaining, self.block_size)
            self.block_usage[block_id] = used
            remaining -= used
        
        self.allocation_log.append(('allocate', seq_id, num_tokens, blocks))
        return True
    
    def append_token(self, seq_id: int) -> bool:
        """Append one token to a sequence (decode step)."""
        if seq_id not in self.block_tables:
            return False
        
        blocks = self.block_tables[seq_id]
        last_block = blocks[-1]
        
        if self.block_usage[last_block] < self.block_size:
            self.block_usage[last_block] += 1
        else:
            # Need a new block
            if not self.free_blocks:
                return False
            new_block = self.free_blocks.pop(0)
            blocks.append(new_block)
            self.block_usage[new_block] = 1
            self.ref_counts[new_block] = 1
        
        return True
    
    def free(self, seq_id: int):
        """Free all blocks for a sequence."""
        if seq_id not in self.block_tables:
            return
        
        for block_id in self.block_tables[seq_id]:
            self.ref_counts[block_id] -= 1
            if self.ref_counts[block_id] <= 0:
                self.free_blocks.append(block_id)
                del self.block_usage[block_id]
                del self.ref_counts[block_id]
        
        del self.block_tables[seq_id]
    
    def fork(self, parent_id: int, child_id: int) -> bool:
        """Copy-on-write fork for beam search."""
        if parent_id not in self.block_tables:
            return False
        
        # Share blocks via reference counting (CoW)
        self.block_tables[child_id] = self.block_tables[parent_id].copy()
        for block_id in self.block_tables[child_id]:
            self.ref_counts[block_id] += 1
        return True
    
    def get_fragmentation(self) -> float:
        """Calculate internal fragmentation."""
        if not self.block_usage:
            return 0.0
        wasted = sum(self.block_size - used for used in self.block_usage.values())
        total = len(self.block_usage) * self.block_size
        return wasted / total

# Test PagedAttention
pa = PagedAttentionSimulator(num_blocks=32, block_size=16)

print("PagedAttention Simulation:")
print(f"  Total blocks: {pa.num_blocks}, Block size: {pa.block_size}")
print(f"  Total KV cache slots: {pa.num_blocks * pa.block_size}")

# Allocate some sequences
pa.allocate(0, 50)   # 50 tokens -> 4 blocks (last block: 2/16 used)
pa.allocate(1, 100)  # 100 tokens -> 7 blocks (last block: 4/16 used)
pa.allocate(2, 30)   # 30 tokens -> 2 blocks (last block: 14/16 used)

print(f"\nAfter allocating 3 sequences:")
print(f"  Used blocks: {pa.used_blocks}/{pa.num_blocks}")
print(f"  Utilization: {pa.utilization:.1%}")
print(f"  Internal fragmentation: {pa.get_fragmentation():.1%}")

for seq_id in pa.block_tables:
    blocks = pa.block_tables[seq_id]
    usage = [pa.block_usage[b] for b in blocks]
    print(f"  Seq {seq_id}: blocks={blocks}, usage={usage}")

In [ ]:
# Demo 2: RadixAttention Simulator

class RadixNode:
    """A node in the radix tree."""
    def __init__(self):
        self.children: Dict[tuple, 'RadixNode'] = {}  # token_key -> child
        self.token_ids: tuple = ()  # tokens stored at this edge
        self.kv_indices: List[int] = []  # GPU memory indices for KV cache
        self.ref_count: int = 0  # Number of active sequences using this node
        self.last_access_time: float = 0.0

class RadixAttentionSimulator:
    """Simulates SGLang's RadixAttention memory management."""
    
    def __init__(self, total_slots: int = 512):
        self.total_slots = total_slots
        self.root = RadixNode()
        self.free_slots: List[int] = list(range(total_slots))
        self.active_sequences: Dict[int, List[RadixNode]] = {}  # seq_id -> [nodes]
        self.time = 0.0
        self.cache_hits = 0
        self.cache_misses = 0
        self.total_tokens_cached = 0
    
    @property
    def used_slots(self):
        return self.total_slots - len(self.free_slots)
    
    @property
    def utilization(self):
        return self.used_slots / self.total_slots
    
    def match_prefix(self, token_ids: List[int]) -> Tuple[int, List[RadixNode]]:
        """Find the longest prefix match in the radix tree."""
        matched_tokens = 0
        matched_nodes = [self.root]
        node = self.root
        
        i = 0
        while i < len(token_ids):
            # Try to find a child that matches
            found = False
            for key, child in node.children.items():
                edge_tokens = child.token_ids
                # Check if edge matches
                match_len = 0
                for j in range(min(len(edge_tokens), len(token_ids) - i)):
                    if edge_tokens[j] == token_ids[i + j]:
                        match_len += 1
                    else:
                        break
                if match_len > 0 and match_len == len(edge_tokens):
                    matched_tokens += match_len
                    matched_nodes.append(child)
                    node = child
                    i += match_len
                    found = True
                    break
            if not found:
                break
        
        return matched_tokens, matched_nodes
    
    def insert(self, seq_id: int, token_ids: List[int]) -> int:
        """Insert a sequence, reusing cached prefix."""
        self.time += 1
        matched_tokens, matched_nodes = self.match_prefix(token_ids)
        
        if matched_tokens > 0:
            self.cache_hits += 1
        else:
            self.cache_misses += 1
        
        # Allocate KV cache for unmatched tokens
        new_tokens = token_ids[matched_tokens:]
        if len(new_tokens) > len(self.free_slots):
            # Need to evict
            self._evict(len(new_tokens) - len(self.free_slots))
        
        if new_tokens and len(self.free_slots) >= len(new_tokens):
            # Create new node for unmatched suffix
            parent = matched_nodes[-1]
            new_node = RadixNode()
            new_node.token_ids = tuple(new_tokens)
            new_node.kv_indices = [self.free_slots.pop(0) for _ in range(len(new_tokens))]
            new_node.ref_count = 1
            new_node.last_access_time = self.time
            
            key = tuple(new_tokens[:min(4, len(new_tokens))])  # Use first few tokens as key
            parent.children[key] = new_node
            matched_nodes.append(new_node)
        
        # Update ref counts
        for node in matched_nodes:
            node.ref_count += 1
            node.last_access_time = self.time
        
        self.active_sequences[seq_id] = matched_nodes
        self.total_tokens_cached = self.used_slots
        
        return matched_tokens  # Number of tokens reused from cache
    
    def release(self, seq_id: int):
        """Release a sequence (decrement ref counts, keep in cache)."""
        if seq_id in self.active_sequences:
            for node in self.active_sequences[seq_id]:
                node.ref_count -= 1
            del self.active_sequences[seq_id]
    
    def _evict(self, slots_needed: int):
        """LRU eviction of unreferenced tree nodes."""
        # Collect all leaf nodes with ref_count == 0
        evictable = []
        self._find_evictable(self.root, evictable)
        evictable.sort(key=lambda n: n.last_access_time)  # LRU
        
        freed = 0
        for node in evictable:
            if freed >= slots_needed:
                break
            self.free_slots.extend(node.kv_indices)
            freed += len(node.kv_indices)
            node.kv_indices = []
            # Remove from parent (simplified)
    
    def _find_evictable(self, node: RadixNode, result: List):
        for child in node.children.values():
            if child.ref_count == 0 and child.kv_indices:
                result.append(child)
            self._find_evictable(child, result)
    
    def get_tree_stats(self) -> Dict:
        """Get statistics about the radix tree."""
        nodes = []
        self._collect_nodes(self.root, nodes)
        return {
            'num_nodes': len(nodes),
            'total_cached_tokens': sum(len(n.token_ids) for n in nodes),
            'active_refs': sum(1 for n in nodes if n.ref_count > 0),
            'evictable': sum(1 for n in nodes if n.ref_count == 0 and n.kv_indices),
        }
    
    def _collect_nodes(self, node: RadixNode, result: List):
        for child in node.children.values():
            result.append(child)
            self._collect_nodes(child, result)

# Test RadixAttention
ra = RadixAttentionSimulator(total_slots=512)

print("RadixAttention Simulation:")
print(f"  Total KV cache slots: {ra.total_slots}")

# Simulate shared prefix scenario
system_prompt = list(range(50))  # 50 token system prompt

reused = ra.insert(0, system_prompt + list(range(100, 130)))  # User 1
print(f"\nRequest 0: {reused} tokens reused from cache")

reused = ra.insert(1, system_prompt + list(range(200, 225)))  # User 2 (shared prefix!)
print(f"Request 1: {reused} tokens reused from cache")

reused = ra.insert(2, system_prompt + list(range(300, 340)))  # User 3 (shared prefix!)
print(f"Request 2: {reused} tokens reused from cache")

print(f"\nUsed slots: {ra.used_slots}/{ra.total_slots}")
print(f"Utilization: {ra.utilization:.1%}")
print(f"Cache hits: {ra.cache_hits}, misses: {ra.cache_misses}")
print(f"Tree stats: {ra.get_tree_stats()}")

In [ ]:
# Demo 3: Visualize memory layouts for both systems

def visualize_memory_layouts():
    """Draw memory layout diagrams for both systems."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 10))
    
    # --- PagedAttention Memory Layout ---
    ax1.set_title('PagedAttention (vLLM) Memory Layout', fontsize=14,
                  fontweight='bold', color='#2196F3')
    ax1.set_xlim(0, 8.5)
    ax1.set_ylim(0, 10)
    ax1.axis('off')
    
    # Draw GPU blocks
    block_colors = {
        'seq0': '#E3F2FD', 'seq1': '#FFF3E0', 'seq2': '#E8F5E9',
        'free': '#F5F5F5', 'waste': '#FFCDD2'
    }
    
    # Physical blocks layout (4x8 grid)
    block_assignments = [
        ('seq0', 16), ('seq0', 16), ('seq0', 16), ('seq0', 2),  # Seq 0: 50 tokens
        ('seq1', 16), ('seq1', 16), ('seq1', 16), ('seq1', 16),  # Seq 1: 100 tokens
        ('seq1', 16), ('seq1', 16), ('seq1', 4), ('seq2', 16),  # Seq 1 cont + Seq 2
        ('seq2', 14), ('free', 0), ('free', 0), ('free', 0),  # Seq 2 cont + free
        ('free', 0), ('free', 0), ('free', 0), ('free', 0),
        ('free', 0), ('free', 0), ('free', 0), ('free', 0),
    ]
    
    for i, (seq, used) in enumerate(block_assignments[:24]):
        row = i // 4
        col = i % 4
        x = col * 2
        y = 9 - row * 1.5
        
        color = block_colors.get(seq, '#F5F5F5')
        rect = FancyBboxPatch((x, y), 1.8, 1.2, boxstyle='round,pad=0.05',
                              facecolor=color, edgecolor='black', linewidth=1)
        ax1.add_patch(rect)
        
        label = f'Block {i}\n'
        if seq == 'free':
            label += 'FREE'
        else:
            label += f'{seq}\n{used}/16'
            if used < 16 and seq != 'free':
                # Show waste
                waste_width = 1.8 * (1 - used/16)
                waste_rect = Rectangle((x + 1.8 - waste_width, y), waste_width, 1.2,
                                       facecolor='#FFCDD2', alpha=0.5)
                ax1.add_patch(waste_rect)
        
        ax1.text(x + 0.9, y + 0.6, label, ha='center', va='center', fontsize=7)
    
    # --- RadixAttention Memory Layout ---
    ax2.set_title('RadixAttention (SGLang) Memory Layout', fontsize=14,
                  fontweight='bold', color='#4CAF50')
    ax2.set_xlim(0, 10)
    ax2.set_ylim(0, 10)
    ax2.axis('off')
    
    # Draw radix tree
    tree_nodes = [
        # (x, y, width, label, color)
        (4, 9, 2, 'Root', '#E8F5E9'),
        (4, 7, 2.5, 'System Prompt\n(50 tokens, shared)', '#C8E6C9'),
        (1, 5, 2, 'User 1 suffix\n(30 tokens)', '#A5D6A7'),
        (4, 5, 2, 'User 2 suffix\n(25 tokens)', '#81C784'),
        (7, 5, 2, 'User 3 suffix\n(40 tokens)', '#66BB6A'),
    ]
    
    for x, y, w, label, color in tree_nodes:
        rect = FancyBboxPatch((x, y), w, 1.5, boxstyle='round,pad=0.1',
                              facecolor=color, edgecolor='#1B5E20', linewidth=2)
        ax2.add_patch(rect)
        ax2.text(x + w/2, y + 0.75, label, ha='center', va='center',
                fontsize=9, fontweight='bold')
    
    # Draw edges
    ax2.annotate('', xy=(5, 8.5), xytext=(5, 9),
                arrowprops=dict(arrowstyle='->', color='#1B5E20', lw=2))
    for end_x in [2, 5, 8]:
        ax2.annotate('', xy=(end_x, 6.5), xytext=(5.25, 7),
                    arrowprops=dict(arrowstyle='->', color='#1B5E20', lw=2))
    
    # Memory comparison text
    ax2.text(5, 3.5, 'Memory Used:\nSystem prompt: 50 slots (shared!)\n'
             'User 1: 30 slots\nUser 2: 25 slots\nUser 3: 40 slots\n'
             'Total: 145 slots (saved 100!)',
             ha='center', va='center', fontsize=10,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    ax1.text(4, 0.3, 'Memory Used: 13 blocks = 208 slots\n'
             'Wasted: 44 slots (internal fragmentation)\n'
             'No prefix sharing between sequences',
             ha='center', va='center', fontsize=10,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    plt.tight_layout()
    plt.savefig('memory_layout_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

visualize_memory_layouts()

In [ ]:
# Demo 4: Simulate cache hit rates for multi-turn conversations

def simulate_multi_turn_cache(num_conversations: int = 10, turns_per_conv: int = 5):
    """Simulate multi-turn conversations and compare cache hit rates."""
    
    system_prompt_len = 50
    user_msg_len = 20
    assistant_msg_len = 40
    block_size = 16
    
    # Track metrics
    pa_compute = []  # Tokens computed (not cached) per request
    ra_compute = []  # Tokens computed (not cached) per request
    pa_total = []
    ra_total = []
    
    # PagedAttention with Automatic Prefix Caching (APC)
    # APC caches at block granularity
    pa_cached_blocks = set()  # hash(block_content) -> cached
    
    # RadixAttention
    ra_cached_prefixes = {}  # prefix_tuple -> length cached
    
    for conv in range(num_conversations):
        system = list(range(system_prompt_len))
        history = system.copy()
        
        for turn in range(turns_per_conv):
            # Add user message
            user_msg = list(range(1000 + conv*100 + turn*20, 1000 + conv*100 + turn*20 + user_msg_len))
            history.extend(user_msg)
            total_tokens = len(history)
            
            # --- PagedAttention APC ---
            # Can only reuse fully-filled blocks
            num_blocks = (total_tokens + block_size - 1) // block_size
            cached_blocks = 0
            for b in range(num_blocks - 1):  # Last block might not be full
                block_content = tuple(history[b*block_size:(b+1)*block_size])
                block_hash = hash(block_content)
                if block_hash in pa_cached_blocks:
                    cached_blocks += 1
                else:
                    pa_cached_blocks.add(block_hash)
            pa_cached_tokens = cached_blocks * block_size
            pa_new_tokens = total_tokens - pa_cached_tokens
            pa_compute.append(pa_new_tokens)
            pa_total.append(total_tokens)
            
            # --- RadixAttention ---
            # Token-level prefix matching
            prefix_key = tuple(history)
            # Find longest cached prefix
            ra_cached = 0
            for length in range(total_tokens, 0, -1):
                sub_prefix = tuple(history[:length])
                if sub_prefix in ra_cached_prefixes:
                    ra_cached = length
                    break
            ra_new_tokens = total_tokens - ra_cached
            ra_compute.append(ra_new_tokens)
            ra_total.append(total_tokens)
            
            # Cache the full sequence for future reuse
            ra_cached_prefixes[prefix_key] = total_tokens
            # Also cache all sub-prefixes
            for l in range(system_prompt_len, total_tokens + 1):
                ra_cached_prefixes[tuple(history[:l])] = l
            
            # Add assistant response to history
            assistant_msg = list(range(2000 + conv*100 + turn*40, 2000 + conv*100 + turn*40 + assistant_msg_len))
            history.extend(assistant_msg)
    
    # Plot results
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. Compute tokens per request
    x = range(len(pa_compute))
    axes[0].plot(x, pa_compute, 'o-', color='#2196F3', label='PagedAttention', alpha=0.7, markersize=4)
    axes[0].plot(x, ra_compute, 's-', color='#4CAF50', label='RadixAttention', alpha=0.7, markersize=4)
    axes[0].set_xlabel('Request Index')
    axes[0].set_ylabel('Tokens to Compute')
    axes[0].set_title('Prefill Computation per Request', fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # 2. Cache hit rate (cumulative)
    pa_hit_rate = [1 - (c/t) for c, t in zip(pa_compute, pa_total)]
    ra_hit_rate = [1 - (c/t) for c, t in zip(ra_compute, ra_total)]
    axes[1].plot(x, pa_hit_rate, 'o-', color='#2196F3', label='PagedAttention', alpha=0.7, markersize=4)
    axes[1].plot(x, ra_hit_rate, 's-', color='#4CAF50', label='RadixAttention', alpha=0.7, markersize=4)
    axes[1].set_xlabel('Request Index')
    axes[1].set_ylabel('Cache Hit Rate')
    axes[1].set_title('Cache Hit Rate Over Time', fontweight='bold')
    axes[1].legend()
    axes[1].set_ylim(0, 1)
    axes[1].grid(True, alpha=0.3)
    
    # 3. Total computation savings
    pa_total_compute = sum(pa_compute)
    ra_total_compute = sum(ra_compute)
    total_possible = sum(pa_total)
    
    savings_data = {
        'No Caching': total_possible,
        'PagedAttention\n(block-level)': pa_total_compute,
        'RadixAttention\n(token-level)': ra_total_compute,
    }
    
    colors = ['#9E9E9E', '#2196F3', '#4CAF50']
    bars = axes[2].bar(savings_data.keys(), savings_data.values(), color=colors, edgecolor='black')
    axes[2].set_ylabel('Total Tokens Computed')
    axes[2].set_title('Total Computation', fontweight='bold')
    for bar, val in zip(bars, savings_data.values()):
        axes[2].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 100,
                    f'{val:,}', ha='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nSummary:")
    print(f"  Total tokens across all requests: {total_possible:,}")
    print(f"  PagedAttention computed: {pa_total_compute:,} ({pa_total_compute/total_possible:.1%})")
    print(f"  RadixAttention computed: {ra_total_compute:,} ({ra_total_compute/total_possible:.1%})")
    print(f"  RadixAttention saves {pa_total_compute - ra_total_compute:,} more tokens than PagedAttention")

simulate_multi_turn_cache()

In [ ]:
# Demo 5: Memory Fragmentation Comparison

def compare_fragmentation():
    """Compare internal fragmentation between both systems."""
    block_size = 16
    
    # Simulate various sequence lengths
    seq_lengths = np.random.randint(10, 500, size=50)
    
    # PagedAttention fragmentation
    pa_fragmentation = []
    for length in seq_lengths:
        num_blocks = (length + block_size - 1) // block_size
        wasted = num_blocks * block_size - length
        frag = wasted / (num_blocks * block_size)
        pa_fragmentation.append(frag)
    
    # RadixAttention fragmentation (minimal - exact allocation)
    ra_fragmentation = [0.0] * len(seq_lengths)  # No internal fragmentation
    
    # But RadixAttention has overhead from tree nodes
    # Approximate: ~8 bytes per node for pointers
    ra_overhead = [8 * (length // 50 + 1) / (length * 2) for length in seq_lengths]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. Fragmentation per sequence
    axes[0].scatter(seq_lengths, pa_fragmentation, alpha=0.6, color='#2196F3',
                   label='PagedAttention', s=40, edgecolor='black')
    axes[0].scatter(seq_lengths, ra_fragmentation, alpha=0.6, color='#4CAF50',
                   label='RadixAttention', s=40, marker='s', edgecolor='black')
    axes[0].set_xlabel('Sequence Length (tokens)')
    axes[0].set_ylabel('Internal Fragmentation')
    axes[0].set_title('Internal Fragmentation vs Sequence Length', fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # 2. Distribution of fragmentation
    axes[1].hist(pa_fragmentation, bins=20, alpha=0.6, color='#2196F3',
                label='PagedAttention', edgecolor='black')
    axes[1].axvline(np.mean(pa_fragmentation), color='#2196F3', linestyle='--',
                   label=f'PA mean: {np.mean(pa_fragmentation):.1%}')
    axes[1].set_xlabel('Fragmentation')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Fragmentation Distribution', fontweight='bold')
    axes[1].legend()
    
    # 3. Effective memory efficiency
    pa_efficiency = [1 - f for f in pa_fragmentation]
    ra_efficiency = [1 - o for o in ra_overhead]
    
    axes[2].scatter(seq_lengths, pa_efficiency, alpha=0.6, color='#2196F3',
                   label='PagedAttention', s=40, edgecolor='black')
    axes[2].scatter(seq_lengths, ra_efficiency, alpha=0.6, color='#4CAF50',
                   label='RadixAttention', s=40, marker='s', edgecolor='black')
    axes[2].set_xlabel('Sequence Length (tokens)')
    axes[2].set_ylabel('Memory Efficiency')
    axes[2].set_title('Effective Memory Efficiency', fontweight='bold')
    axes[2].legend()
    axes[2].set_ylim(0.8, 1.02)
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nPagedAttention avg fragmentation: {np.mean(pa_fragmentation):.2%}")
    print(f"PagedAttention max fragmentation: {max(pa_fragmentation):.2%}")
    print(f"RadixAttention avg tree overhead: {np.mean(ra_overhead):.2%}")
    print(f"\nPagedAttention wastes space on partially filled last blocks.")
    print(f"RadixAttention has minimal overhead but requires tree maintenance.")

compare_fragmentation()

In [ ]:
# Demo 6: Prefix sharing efficiency comparison

def compare_prefix_sharing():
    """Compare how efficiently each system shares prefixes."""
    block_size = 16
    
    scenarios = [
        {
            'name': 'Chatbot (shared system prompt)',
            'system_prompt': 100,
            'num_users': 20,
            'user_msg_range': (20, 60),
        },
        {
            'name': 'Few-shot learning (shared examples)',
            'system_prompt': 500,
            'num_users': 10,
            'user_msg_range': (30, 80),
        },
        {
            'name': 'RAG (varied prefixes)',
            'system_prompt': 50,
            'num_users': 15,
            'user_msg_range': (100, 300),
        },
        {
            'name': 'Code completion (project context)',
            'system_prompt': 2000,
            'num_users': 5,
            'user_msg_range': (10, 30),
        },
    ]
    
    results = []
    for scenario in scenarios:
        sp = scenario['system_prompt']
        n = scenario['num_users']
        
        # Without sharing: each request stores full prefix
        user_tokens = [random.randint(*scenario['user_msg_range']) for _ in range(n)]
        no_sharing = sum(sp + ut for ut in user_tokens)
        
        # PagedAttention (block-level sharing)
        shared_blocks = sp // block_size  # Full blocks that can be shared
        shared_tokens = shared_blocks * block_size
        pa_memory = shared_tokens + sum(  # shared prefix
            ((sp - shared_tokens + ut + block_size - 1) // block_size) * block_size
            for ut in user_tokens
        )
        
        # RadixAttention (token-level sharing)
        ra_memory = sp + sum(user_tokens)  # Exact sharing
        
        results.append({
            'name': scenario['name'],
            'no_sharing': no_sharing,
            'paged': pa_memory,
            'radix': ra_memory,
        })
    
    # Plot
    fig, ax = plt.subplots(figsize=(14, 6))
    x = np.arange(len(results))
    width = 0.25
    
    bars1 = ax.bar(x - width, [r['no_sharing'] for r in results], width,
                  label='No Sharing', color='#9E9E9E', edgecolor='black')
    bars2 = ax.bar(x, [r['paged'] for r in results], width,
                  label='PagedAttention', color='#2196F3', edgecolor='black')
    bars3 = ax.bar(x + width, [r['radix'] for r in results], width,
                  label='RadixAttention', color='#4CAF50', edgecolor='black')
    
    ax.set_ylabel('Total Memory (token slots)')
    ax.set_title('Memory Usage by Prefix Sharing Strategy', fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([r['name'] for r in results], rotation=15, ha='right')
    ax.legend()
    
    # Add savings annotations
    for i, r in enumerate(results):
        savings = (1 - r['radix'] / r['no_sharing']) * 100
        ax.text(i + width, r['radix'] + r['no_sharing'] * 0.02,
               f'-{savings:.0f}%', ha='center', fontsize=9, color='#2E7D32', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\nMemory Savings Summary:")
    print(f"{'Scenario':<35} {'PA Savings':<15} {'RA Savings':<15}")
    print('=' * 65)
    for r in results:
        pa_save = (1 - r['paged'] / r['no_sharing']) * 100
        ra_save = (1 - r['radix'] / r['no_sharing']) * 100
        print(f"{r['name']:<35} {pa_save:<15.1f}% {ra_save:<15.1f}%")

compare_prefix_sharing()

In [ ]:
# Demo 7: Copy-on-Write for beam search (PagedAttention advantage)

def demonstrate_cow_beam_search():
    """Show how PagedAttention handles beam search efficiently."""
    pa = PagedAttentionSimulator(num_blocks=32, block_size=16)
    
    # Initial sequence
    pa.allocate(0, 64)  # Parent: 64 tokens
    print("Beam Search with Copy-on-Write:")
    print(f"  Parent (seq 0): 64 tokens, {pa.block_tables[0]}")
    print(f"  Blocks used: {pa.used_blocks}")
    
    # Fork into 4 beams
    for beam in range(1, 5):
        pa.fork(0, beam)
    
    print(f"\nAfter forking into 4 beams:")
    print(f"  Blocks used: {pa.used_blocks} (shared via CoW!)")
    for seq_id in pa.block_tables:
        print(f"  Seq {seq_id}: blocks={pa.block_tables[seq_id]}, "
              f"ref_counts={[pa.ref_counts[b] for b in pa.block_tables[seq_id]]}")
    
    # Each beam generates different tokens
    print(f"\nEach beam appends tokens:")
    for beam in range(1, 5):
        for _ in range(3):  # 3 decode steps
            pa.append_token(beam)
    
    print(f"  Blocks used after decode: {pa.used_blocks}")
    print(f"  Shared blocks are NOT copied until divergence!")
    
    # Visualize
    fig, ax = plt.subplots(figsize=(12, 5))
    
    # Timeline of block usage
    stages = ['Initial\n(1 seq)', 'After Fork\n(4 beams)', 'After 3\nDecode Steps']
    without_cow = [4, 16, 20]  # Without CoW: full copy for each beam
    with_cow = [4, 4, 8]  # With CoW: share blocks, only allocate new ones
    
    x = np.arange(len(stages))
    width = 0.35
    ax.bar(x - width/2, without_cow, width, label='Without CoW', color='#F44336', alpha=0.8)
    ax.bar(x + width/2, with_cow, width, label='With CoW (PagedAttention)', color='#4CAF50', alpha=0.8)
    
    ax.set_ylabel('Blocks Used')
    ax.set_title('Beam Search Memory: Copy-on-Write Savings', fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(stages)
    ax.legend()
    
    for bars_group in [ax.containers[0], ax.containers[1]]:
        for bar in bars_group:
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
                   f'{int(bar.get_height())}', ha='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

demonstrate_cow_beam_search()

---
## Hands-on Assignment 1: Implement Both Memory Managers as Simulators

Extend the simulators to handle:
1. **PagedAttention**: Add automatic prefix caching (APC) support
2. **RadixAttention**: Add proper tree splitting when partial prefix matches occur
3. Run both on the same workload and compare

In [ ]:
# Assignment 1: Enhanced simulators

class PagedAttentionWithAPC(PagedAttentionSimulator):
    """PagedAttention with Automatic Prefix Caching."""
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # APC: hash(block_tokens) -> block_id (cached blocks)
        self.prefix_cache: Dict[int, int] = {}  # content_hash -> block_id
        self.cache_hits = 0
        self.cache_misses = 0
    
    def allocate_with_prefix(self, seq_id: int, token_ids: List[int]) -> int:
        """Allocate blocks, reusing cached prefix blocks.
        
        TODO: Implement APC logic:
        1. Divide token_ids into blocks of block_size
        2. For each full block, compute hash of its contents
        3. If hash exists in prefix_cache, reuse that block (increment ref_count)
        4. Otherwise, allocate a new block and add to cache
        5. Return number of tokens reused from cache
        """
        # TODO: Your implementation here
        reused_tokens = 0
        return reused_tokens


class EnhancedRadixCache(RadixAttentionSimulator):
    """Enhanced RadixAttention with proper tree splitting."""
    
    def split_node(self, parent: RadixNode, child: RadixNode, split_pos: int):
        """Split a node when partial prefix match occurs.
        
        TODO: Implement node splitting:
        1. Create a new intermediate node with tokens[:split_pos]
        2. Move remaining tokens to a new child
        3. Redistribute KV indices between the two nodes
        4. Update parent's children dict
        """
        # TODO: Your implementation here
        pass


# Test template
print("Implement the enhanced simulators and test with this workload:")
print("\nTest workload:")
test_prompts = [
    list(range(100)),            # Request 0: [0, 1, ..., 99]
    list(range(80)) + list(range(200, 230)),  # Request 1: shares 80-token prefix
    list(range(100)) + list(range(300, 320)),  # Request 2: shares full 100 prefix
    list(range(50)) + list(range(400, 480)),   # Request 3: shares 50-token prefix
]

for i, prompt in enumerate(test_prompts):
    print(f"  Request {i}: {len(prompt)} tokens, prefix overlap with req 0: {len(set(range(100)) & set(prompt[:100]))}")

---
## Hands-on Assignment 2: Benchmark Cache Hit Rates with ShareGPT Dataset

Simulate a realistic workload based on ShareGPT-like conversations.

In [ ]:
# Assignment 2: ShareGPT-like workload simulation

def generate_sharegpt_workload(num_conversations: int = 20,
                                max_turns: int = 6,
                                system_prompt_len: int = 80):
    """Generate a ShareGPT-like workload with multi-turn conversations."""
    workload = []
    system_prompt = list(range(system_prompt_len))
    
    for conv_id in range(num_conversations):
        num_turns = random.randint(2, max_turns)
        history = system_prompt.copy()
        
        for turn in range(num_turns):
            user_len = random.randint(10, 100)
            user_msg = list(range(10000 + conv_id * 1000 + turn * 200,
                                  10000 + conv_id * 1000 + turn * 200 + user_len))
            history.extend(user_msg)
            
            workload.append({
                'conv_id': conv_id,
                'turn': turn,
                'token_ids': history.copy(),
                'total_tokens': len(history),
            })
            
            # Add assistant response to history for next turn
            asst_len = random.randint(20, 150)
            asst_msg = list(range(50000 + conv_id * 1000 + turn * 200,
                                  50000 + conv_id * 1000 + turn * 200 + asst_len))
            history.extend(asst_msg)
    
    return workload

sharegpt = generate_sharegpt_workload()
print(f"Generated {len(sharegpt)} requests from {len(set(r['conv_id'] for r in sharegpt))} conversations")

# TODO: Run both memory managers on this workload and compare:
# 1. Cache hit rate per request
# 2. Total memory used
# 3. Tokens saved from caching
# 4. Plot results

print("\nTODO: Implement the benchmark and plot cache hit rates.")
print("Expected: RadixAttention should show 20-40% higher cache hit rate")
print("on multi-turn conversations due to token-level prefix matching.")

---
## Hands-on Assignment 3: Design a Workload that Maximizes RadixAttention Advantage

Find the workload characteristics that give RadixAttention the biggest edge.

In [ ]:
# Assignment 3: Maximize RadixAttention advantage

def analyze_advantage_factors():
    """Analyze which factors most influence the RadixAttention advantage."""
    
    factors = {
        'System Prompt Length': [10, 50, 100, 200, 500, 1000],
        'Number of Users': [2, 5, 10, 20, 50],
        'Prefix Overlap Ratio': [0.0, 0.25, 0.5, 0.75, 1.0],
    }
    
    block_size = 16
    user_msg_len = 40
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Factor 1: System prompt length
    sp_lengths = factors['System Prompt Length']
    pa_savings = []
    ra_savings = []
    for sp_len in sp_lengths:
        n_users = 10
        no_share = n_users * (sp_len + user_msg_len)
        pa = (sp_len // block_size) * block_size + n_users * ((sp_len % block_size + user_msg_len + block_size - 1) // block_size) * block_size
        ra = sp_len + n_users * user_msg_len
        pa_savings.append(1 - pa / no_share)
        ra_savings.append(1 - ra / no_share)
    
    axes[0].plot(sp_lengths, pa_savings, 'o-', color='#2196F3', label='PagedAttention')
    axes[0].plot(sp_lengths, ra_savings, 's-', color='#4CAF50', label='RadixAttention')
    axes[0].set_xlabel('System Prompt Length')
    axes[0].set_ylabel('Memory Savings')
    axes[0].set_title('Impact of System Prompt Length', fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Factor 2: Number of users
    user_counts = factors['Number of Users']
    pa_savings2 = []
    ra_savings2 = []
    sp_len = 200
    for n in user_counts:
        no_share = n * (sp_len + user_msg_len)
        pa = (sp_len // block_size) * block_size + n * ((sp_len % block_size + user_msg_len + block_size - 1) // block_size) * block_size
        ra = sp_len + n * user_msg_len
        pa_savings2.append(1 - pa / no_share)
        ra_savings2.append(1 - ra / no_share)
    
    axes[1].plot(user_counts, pa_savings2, 'o-', color='#2196F3', label='PagedAttention')
    axes[1].plot(user_counts, ra_savings2, 's-', color='#4CAF50', label='RadixAttention')
    axes[1].set_xlabel('Number of Concurrent Users')
    axes[1].set_ylabel('Memory Savings')
    axes[1].set_title('Impact of User Count', fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Factor 3: RadixAttention advantage gap
    advantage_gap = [ra - pa for pa, ra in zip(pa_savings, ra_savings)]
    axes[2].bar(range(len(sp_lengths)), advantage_gap, color='#FF9800', edgecolor='black')
    axes[2].set_xticks(range(len(sp_lengths)))
    axes[2].set_xticklabels([str(s) for s in sp_lengths])
    axes[2].set_xlabel('System Prompt Length')
    axes[2].set_ylabel('RadixAttention Advantage')
    axes[2].set_title('RadixAttention Advantage Gap', fontweight='bold')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\nDesign your optimal RadixAttention workload:")
    print("  TODO: Specify the exact parameters that maximize RA advantage")
    print("  Hint: Consider long system prompts + many users + multi-turn")

analyze_advantage_factors()

# TODO: Design your workload here
optimal_workload = {
    'system_prompt_length': 'TODO',
    'num_users': 'TODO',
    'turns_per_user': 'TODO',
    'user_msg_length': 'TODO',
    'expected_ra_advantage': 'TODO%',
    'justification': 'TODO: explain why this maximizes the advantage',
}
print("\nYour optimal workload:", json.dumps(optimal_workload, indent=2))

In [ ]:
import json

print("\n--- End of Chapter 10.3 ---")
print("Next: Chapter 10.4 - Performance Benchmarks")
print("\nKey takeaways:")
print("  1. PagedAttention: great for beam search (CoW), block-level sharing")
print("  2. RadixAttention: superior prefix sharing, minimal fragmentation")
print("  3. The advantage gap grows with shared prefix length and user count")

## Summary

| Feature | PagedAttention (vLLM) | RadixAttention (SGLang) |
|---------|----------------------|------------------------|
| **Sharing Granularity** | Block-level (16 tokens) | Token-level (exact) |
| **Fragmentation** | Internal (last block waste) | Minimal |
| **Beam Search** | Excellent (CoW) | Good |
| **Prefix Caching** | Block-aligned APC | Tree-based automatic |
| **Eviction** | Full sequence | LRU on tree nodes |
| **Complexity** | Moderate | Higher (tree maintenance) |